In [1]:
# 🧹 1. Clear out any corrupted, half-finished downloads
!rm -rf ~/.keras/datasets/cifar-10-*
!mkdir -p ~/.keras/datasets

# 🚀 2. Force download directly using Linux (bypasses TensorFlow's downloader)
print("Downloading CIFAR-10 directly (this should take ~5 seconds)...")
!wget -q --show-progress -O ~/.keras/datasets/cifar-10-batches-py.tar.gz https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz
print("✅ Download complete! You can now run your main Python cell.")

-10-batches-py.tar.  10%[=>                  ]  17.05M  93.6KB/s    eta 26m 28s^C
✅ Download complete! You can now run your main Python cell.


In [2]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from scipy.stats import mode
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print(f"🚀 TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# 1. Load Data (FASHION-MNIST - Ultra Fast Server!)
print("\n📥 Loading Fashion-MNIST Dataset...")
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# 2. Normalize, Reshape (Add grayscale channel), and Split
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train_full = np.expand_dims(x_train_full, -1)
x_test = np.expand_dims(x_test, -1)

x_train, x_val, y_train, y_val = train_test_split(x_train_full, y_train_full, test_size=0.15, random_state=42)

# 3. Data Augmentation Layer
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

print(f"✅ Training images: {x_train.shape[0]} | Validation: {x_val.shape[0]} | Test: {x_test.shape[0]}")

🚀 TensorFlow Version: 2.20.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

📥 Loading Fashion-MNIST Dataset...
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✅ Training images: 51000 | Validation: 9000 | Test: 10000


In [3]:
BATCH_SIZE = 32
MAX_EPOCHS = 30
LEARNING_RATE = 0.001

import os
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

def plot_history(history, title, filename):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train')
    plt.plot(history.history['val_accuracy'], label='Validation')
    plt.title(f'{title} - Accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train')
    plt.plot(history.history['val_loss'], label='Validation')
    plt.title(f'{title} - Loss')
    plt.legend()
    plt.savefig(f'results/{filename}')
    plt.close()

# --- CNN 1 (Baseline) ---
print("\n🏗️ Building & Training CNN 1 (Baseline)...")
inputs = layers.Input(shape=(28, 28, 1)) # Updated for Fashion-MNIST shape!
x = data_augmentation(inputs)
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Flatten()(x)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(10, activation='softmax')(x)

cnn1 = models.Model(inputs, outputs)
cnn1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist1 = cnn1.fit(x_train, y_train, epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, validation_data=(x_val, y_val),
                 callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True), callbacks.ModelCheckpoint('models/cnn_baseline.keras', save_best_only=True)], verbose=1)
plot_history(hist1, "CNN 1 (Baseline)", "training_history_cnn1.png")

# --- CNN 2 (Regularized) ---
print("\n🏗️ Building & Training CNN 2 (Regularized)...")
inputs = layers.Input(shape=(28, 28, 1))
x = data_augmentation(inputs)
x = layers.Conv2D(32, (3, 3), padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x); x = layers.MaxPooling2D((2, 2))(x); x = layers.Dropout(0.3)(x)
x = layers.Conv2D(64, (3, 3), padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x); x = layers.MaxPooling2D((2, 2))(x); x = layers.Dropout(0.4)(x)
x = layers.Flatten()(x); x = layers.Dense(128, activation='relu')(x); x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation='softmax')(x)

cnn2 = models.Model(inputs, outputs)
cnn2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist2 = cnn2.fit(x_train, y_train, epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, validation_data=(x_val, y_val),
                 callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True), callbacks.ModelCheckpoint('models/cnn_regularized.keras', save_best_only=True)], verbose=1)
plot_history(hist2, "CNN 2 (Regularized)", "training_history_cnn2.png")

# --- CNN 3 (Deep) ---
print("\n🏗️ Building & Training CNN 3 (Deep)...")
inputs = layers.Input(shape=(28, 28, 1))
x = data_augmentation(inputs)
x = layers.Conv2D(64, (3, 3), padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
x = layers.Conv2D(64, (3, 3), padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x); x = layers.MaxPooling2D((2, 2))(x); x = layers.Dropout(0.3)(x)
x = layers.Conv2D(128, (3, 3), padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
x = layers.Conv2D(128, (3, 3), padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x); x = layers.MaxPooling2D((2, 2))(x); x = layers.Dropout(0.4)(x)
x = layers.GlobalAveragePooling2D()(x); x = layers.Dense(128, activation='relu')(x); x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation='softmax')(x)

cnn3 = models.Model(inputs, outputs)
cnn3.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist3 = cnn3.fit(x_train, y_train, epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, validation_data=(x_val, y_val),
                 callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True), callbacks.ModelCheckpoint('models/cnn_deep.keras', save_best_only=True)], verbose=1)
plot_history(hist3, "CNN 3 (Deep)", "training_history_cnn3.png")
print("\n✅ All 3 CNN models trained and saved!")


🏗️ Building & Training CNN 1 (Baseline)...
Epoch 1/30
1594/1594 ━━━━━━━━━━━━━━━━━━━━ 17s 7ms/step - accuracy: 0.7571 - loss: 0.6632 - val_accuracy: 0.8260 - val_loss: 0.4870
Epoch 2/30
1594/1594 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.8251 - loss: 0.4765 - val_accuracy: 0.8023 - val_loss: 0.5415
Epoch 3/30
1594/1594 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.8454 - loss: 0.4198 - val_accuracy: 0.8551 - val_loss: 0.3949
Epoch 4/30
1594/1594 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.8575 - loss: 0.3861 - val_accuracy: 0.8668 - val_loss: 0.3666
Epoch 5/30
1594/1594 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.8667 - loss: 0.3629 - val_accuracy: 0.8752 - val_loss: 0.3487
Epoch 6/30
1594/1594 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.8737 - loss: 0.3430 - val_accuracy: 0.8683 - val_loss: 0.3555
Epoch 7/30
1594/1594 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.8783 - loss: 0.3297 - val_accuracy: 0.8741 - val_loss: 0.3589
Epoch 8/30
1594/1594 ━━━━━━━━━━━━━━━━━━

In [4]:
print("\n" + "="*50)
print("EVALUATION, BENCHMARKS, & ROBUSTNESS")
print("="*50)

# Load Models & Predict
cnn1 = tf.keras.models.load_model('models/cnn_baseline.keras')
cnn2 = tf.keras.models.load_model('models/cnn_regularized.keras')
cnn3 = tf.keras.models.load_model('models/cnn_deep.keras')

p1 = cnn1.predict(x_test, verbose=0)
p2 = cnn2.predict(x_test, verbose=0)
p3 = cnn3.predict(x_test, verbose=0)

c1 = np.argmax(p1, axis=1)
c2 = np.argmax(p2, axis=1)
c3 = np.argmax(p3, axis=1)

# Ensembles
p_soft = (p1 + p2 + p3) / 3.0
c_soft = np.argmax(p_soft, axis=1)
c_hard, _ = mode(np.vstack((c1, c2, c3)), axis=0, keepdims=False)

# 1. Metrics
def get_m(yt, yp): return [accuracy_score(yt, yp), f1_score(yt, yp, average='weighted', zero_division=0)]
metrics_df = pd.DataFrame({
    'CNN 1': get_m(y_test, c1), 'CNN 2': get_m(y_test, c2), 'CNN 3': get_m(y_test, c3),
    'Ens (Hard)': get_m(y_test, c_hard), 'Ens (Soft)': get_m(y_test, c_soft)
}, index=['Accuracy', 'F1-Score']).T

# 2. Benchmarks
def bench(m, d):
    _ = m.predict(d[:10], verbose=0)
    t0 = time.time(); _ = m.predict(d, verbose=0); t1 = time.time()
    avg_t = (t1 - t0); return (avg_t / len(d)) * 1000, len(d) / avg_t

l1, t1 = bench(cnn1, x_test[:1000]); l2, t2 = bench(cnn2, x_test[:1000]); l3, t3 = bench(cnn3, x_test[:1000])
l_ens = l1 + l2 + l3; t_ens = 1000 / ((l_ens / 1000) * 1000)
s1 = os.path.getsize('models/cnn_baseline.keras')/(1024*1024); s2 = os.path.getsize('models/cnn_regularized.keras')/(1024*1024); s3 = os.path.getsize('models/cnn_deep.keras')/(1024*1024)

bench_df = pd.DataFrame({
    'Size (MB)': [s1, s2, s3, s1+s2+s3, s1+s2+s3],
    'Latency (ms)': [l1, l2, l3, l_ens, l_ens],
    'Throughput (img/s)': [t1, t2, t3, t_ens, t_ens]
}, index=['CNN 1', 'CNN 2', 'CNN 3', 'Ens (Hard)', 'Ens (Soft)'])

# 3. Robustness
xt_sub = x_test[:1000]; yt_sub = y_test[:1000]
test_sets = {'Original': xt_sub, 'Rotated': np.rot90(xt_sub, 1, (1, 2)), 'Noisy': np.clip(xt_sub + np.random.normal(0, 0.1, xt_sub.shape), 0, 1)}

rob = {}
for name, data in test_sets.items():
    rob[name] = [
        accuracy_score(yt_sub, np.argmax(cnn1.predict(data, verbose=0), axis=1)),
        accuracy_score(yt_sub, np.argmax(cnn2.predict(data, verbose=0), axis=1)),
        accuracy_score(yt_sub, np.argmax(cnn3.predict(data, verbose=0), axis=1)),
        accuracy_score(yt_sub, np.argmax((cnn1.predict(data, verbose=0)+cnn2.predict(data, verbose=0)+cnn3.predict(data, verbose=0))/3, axis=1))
    ]
rob_df = pd.DataFrame(rob, index=['CNN 1', 'CNN 2', 'CNN 3', 'Ens (Soft)']).T

print("\n--- METRICS ---"); display(metrics_df.style.format("{:.2%}"))
print("\n--- BENCHMARKS ---"); display(bench_df.style.format("{:.2f}"))
print("\n--- ROBUSTNESS ---"); display(rob_df.style.format("{:.2%}"))

metrics_df.to_csv('results/evaluation_metrics.csv')
bench_df.to_csv('results/production_benchmarks.csv')
rob_df.to_csv('results/robustness_results.csv')
print("\n✅ All project code is complete! Check the 'results/' folder for your CSV files.")


EVALUATION, BENCHMARKS, & ROBUSTNESS

--- METRICS ---


,Accuracy,F1-Score
CNN 1,89.97%,89.83%
CNN 2,87.39%,87.23%
CNN 3,90.01%,90.09%
Ens (Hard),90.25%,90.16%
Ens (Soft),90.70%,90.63%



--- BENCHMARKS ---


,Size (MB),Latency (ms),Throughput (img/s)
CNN 1,2.57,0.31,3248.34
CNN 2,4.90,0.46,2177.50
CNN 3,3.28,0.29,3423.78
Ens (Hard),10.75,1.06,944.14
Ens (Soft),10.75,1.06,944.14



--- ROBUSTNESS ---


,CNN 1,CNN 2,CNN 3,Ens (Soft)
Original,90.40%,88.60%,90.80%,91.10%
Rotated,5.90%,5.60%,9.70%,6.70%
Noisy,74.50%,79.40%,22.90%,68.50%



✅ All project code is complete! Check the 'results/' folder for your CSV files.
